In [1]:
import numpy as np
import keras
from keras import layers
import os
import cv2
import matplotlib.pyplot as plt


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wanderdust/coin-images")

print("Path to dataset files:", path)

C:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|███████████████████████████████████████████████████████████████████████████████| 459M/459M [00:23<00:00, 20.9MB/s]

Extracting files...


Path to dataset files: C:\Users\USER\.cache\kagglehub\datasets\wanderdust\coin-images\versions\1


In [3]:
inputs = keras.Input(shape=(784,))

In [7]:
dense = layers.Dense(64, activation="relu")
x = dense(inputs)

In [8]:
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(10)(x)

In [20]:
dataset_path = r"C:\Users\USER\.cache\kagglehub\datasets\wanderdust\coin-images\versions\1"

for root, dirs, files in os.walk(dataset_path):
    depth = root.replace(dataset_path, "").count(os.sep)
    if depth <= 3:
        print(f"{'  ' * depth}{os.path.basename(root)}/  ({len(files)} files)")

1/  (1 files)
  coins/  (0 files)
    data/  (1 files)
      test/  (0 files)
      train/  (0 files)
      validation/  (0 files)


In [21]:
import os, json, glob

json_path = glob.glob(os.path.join(dataset_path, "**", "cat_to_name.json"), recursive=True)
print(json_path)

with open(json_path[0]) as f:
    cat_to_name = json.load(f)

['C:\\Users\\USER\\.cache\\kagglehub\\datasets\\wanderdust\\coin-images\\versions\\1\\cat_to_name.json', 'C:\\Users\\USER\\.cache\\kagglehub\\datasets\\wanderdust\\coin-images\\versions\\1\\coins\\data\\cat_to_name.json']


In [22]:
data_dir = os.path.join(dataset_path, "coins", "data")
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "validation")
test_dir  = os.path.join(data_dir, "test")

classes = sorted(os.listdir(train_dir))
print(len(classes), classes[:5])
print(len(os.listdir(os.path.join(train_dir, classes[0]))), "images in first class")

211 ['1', '10', '100', '101', '102']
28 images in first class


In [23]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Adjust if your structure differs (e.g. there may be an extra "data" folder)
data_dir = os.path.join(dataset_path, "coins", "data")
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "validation")   # may be "valid"
test_dir  = os.path.join(data_dir, "test")

IMG_SIZE = 224
BATCH_SIZE = 32 #how mant images go through the model at finetuning. How many are we looking at before updating the weights,
#batch size is to look at when were looking at training time and resources

# ImageNet stats - use these if you plan to use a pretrained model
#Noramlise computes (pixel - mean) / std
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

#rotate, flip, and color jitter make the image look a bit different every time to make the model learn what a coin looks like, rrather than what the images look like
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),          # coins can be any orientation
    transforms.RandomHorizontalFlip(), # play with this. coins dont flip
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), #converts pixels from 0-255 to 0-1 and rearranged image into a tensor
    transforms.Normalize(mean, std), #shifts and scales the tensor
])

#we don't need random augmentation for the evaluation and testing sets
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
#the transforms are stored in the ImageFolder and applied each time the image is loaded. This ensures the random augmentations are different every epoch
train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir,   transform=eval_transform)
test_ds  = datasets.ImageFolder(test_dir,  transform=eval_transform)

#wraps a dataset and hands it off to the model, armed with all the level settings we decided on
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Classes:", len(train_ds.classes)) #counts number of classes
print("Train/Val/Test sizes:", len(train_ds), len(val_ds), len(test_ds))

Classes: 211
Train/Val/Test sizes: 6413 844 844


In [25]:
from collections import Counter
counts = Counter(train_ds.targets)
print(min(counts.values()), max(counts.values()))

8 52


In [26]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [28]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(36)

In [30]:
import torch.nn as nn
from torchvision import models

num_classes = len(train_ds.classes)

#resnet is pretrained on imagenet, so it knows to recognize features of an image
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for p in model.parameters():
    #requires_grad flags each parameter on whether or not to compute gradients for it.
    #were setting it to false for now so that they're excluded from learning to keep pretrained knowledge in tact and save cpu
    #consider playing with it
    p.requires_grad = False          # freeze the pretrained backbone

#model fc (fully connected) is the last layer of resnet, turns the features into class scores
#this is the only part that learns
model.fc = nn.Linear(model.fc.in_features, num_classes)   # new head for 211 coins
model = model.to(device)

#how wrong is the model? 
criterion = nn.CrossEntropyLoss()
#tries to minimize the wrongness
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [32]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader: #x - images, a tensor shaped [32, 3, 224, 224], y - true labels, tensor or 32 class indices
            x, y = x.to(device), y.to(device) #both sent to device
            out = model(x)
            loss = criterion(out, y) #average loss per image in batch
            if train:
                optimizer.zero_grad() #clear the old gradients so they dont confuse the new calculations
                loss.backward() #compute new gradients (backpropagation)
                optimizer.step() #updates weights
            total_loss += loss.item() * x.size(0) #average losses * images in the batch = total losses
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

for epoch in range(5):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f}")

Epoch 1: train acc 0.272 | val loss 3.571 | val acc 0.448
Epoch 2: train acc 0.634 | val loss 2.794 | val acc 0.575
Epoch 3: train acc 0.738 | val loss 2.350 | val acc 0.614
Epoch 4: train acc 0.792 | val loss 2.047 | val acc 0.654
Epoch 5: train acc 0.831 | val loss 1.860 | val acc 0.669


In [42]:
results = {}
results["with_flip"] = {"val_acc": va, "val_loss": vl}
print(results)

{'with_flip': {'val_acc': 0.669431279620853, 'val_loss': 1.8604557898372271}}


This time we'll run everything exactly the same, except with no horizontal flips

In [35]:
# Adjust if your structure differs (e.g. there may be an extra "data" folder)
data_dir = os.path.join(dataset_path, "coins", "data")
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "validation")   # may be "valid"
test_dir  = os.path.join(data_dir, "test")

IMG_SIZE = 224
BATCH_SIZE = 32 #how mant images go through the model at finetuning. How many are we looking at before updating the weights,
#batch size is to look at when were looking at training time and resources

# ImageNet stats - use these if you plan to use a pretrained model
#Noramlise computes (pixel - mean) / std
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

#rotate, flip, and color jitter make the image look a bit different every time to make the model learn what a coin looks like, rrather than what the images look like
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),          # coins can be any orientation
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), #converts pixels from 0-255 to 0-1 and rearranged image into a tensor
    transforms.Normalize(mean, std), #shifts and scales the tensor
])

#we don't need random augmentation for the evaluation and testing sets
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
#the transforms are stored in the ImageFolder and applied each time the image is loaded. This ensures the random augmentations are different every epoch
train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir,   transform=eval_transform)
test_ds  = datasets.ImageFolder(test_dir,  transform=eval_transform)

#wraps a dataset and hands it off to the model, armed with all the level settings we decided on
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Classes:", len(train_ds.classes)) #counts number of classes
print("Train/Val/Test sizes:", len(train_ds), len(val_ds), len(test_ds))

Classes: 211
Train/Val/Test sizes: 6413 844 844


In [36]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(36)

In [37]:
num_classes = len(train_ds.classes)

#resnet is pretrained on imagenet, so it knows to recognize features of an image
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for p in model.parameters():
    #requires_grad flags each parameter on whether or not to compute gradients for it.
    #were setting it to false for now so that they're excluded from learning to keep pretrained knowledge in tact and save cpu
    #consider playing with it
    p.requires_grad = False          # freeze the pretrained backbone

#model fc (fully connected) is the last layer of resnet, turns the features into class scores
#this is the only part that learns
model.fc = nn.Linear(model.fc.in_features, num_classes)   # new head for 211 coins
model = model.to(device)

#how wrong is the model? 
criterion = nn.CrossEntropyLoss()
#tries to minimize the wrongness
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [38]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader: #x - images, a tensor shaped [32, 3, 224, 224], y - true labels, tensor or 32 class indices
            x, y = x.to(device), y.to(device) #both sent to device
            out = model(x)
            loss = criterion(out, y) #average loss per image in batch
            if train:
                optimizer.zero_grad() #clear the old gradients so they dont confuse the new calculations
                loss.backward() #compute new gradients (backpropagation)
                optimizer.step() #updates weights
            total_loss += loss.item() * x.size(0) #average losses * images in the batch = total losses
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

for epoch in range(5):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f}")

Epoch 1: train acc 0.274 | val loss 3.549 | val acc 0.457
Epoch 2: train acc 0.645 | val loss 2.696 | val acc 0.556
Epoch 3: train acc 0.754 | val loss 2.320 | val acc 0.608
Epoch 4: train acc 0.805 | val loss 2.015 | val acc 0.661
Epoch 5: train acc 0.840 | val loss 1.840 | val acc 0.659


In [41]:
results["no_flip"] = {"val_acc": va, "val_loss": vl}
print(results)

{'with_flip': {'val_acc': 0.6587677725118484, 'val_loss': 1.8402982469983575}, 'no_flip': {'val_acc': 0.6587677725118484, 'val_loss': 1.8402982469983575}}


In [ ]:
name = input("enter name")

In [15]:
model = keras.Model(inputs=inputs, outputs=outputs, name=dataset_path)

In [16]:
model.summary()

Model: "C:\Users\USER\.cache\kagglehub\datasets\wanderdust\coin-images\versions\1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │          50,240 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 55,050 (215.04 KB)

 Trainable params: 55,050 (215.04 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.reshape(60000, 784).astype("float32") / 255
x_test = x_test.reshape(10000, 784).astype("float32") / 255

model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history = model.fit(x_train, y_train, batch_size=64, epochs=2, validation_split=0.2)

test_scores = model.evaluate(x_test, y_test, verbose=2)
print("Test loss:", test_scores[0])
print("Test accuracy:", test_scores[1])

Epoch 1/2


ValueError: 'C:\Users\USER\.cache\kagglehub\datasets\wanderdust\coin-images\versions\1' is not a valid root scope name. A root scope name has to match the following pattern: ^[A-Za-z0-9.][A-Za-z0-9_.\\/>-]*$

In [ ]:
model.save("my_model.keras")
del model
# Recreate the exact same model purely from the file:
model = keras.models.load_model("my_model.keras")

In [ ]:
encoder_input = keras.Input(shape=(28, 28, 1), name="img")
x = layers.Conv2D(16, 3, activation="relu")(encoder_input)
x = layers.Conv2D(32, 3, activation="relu")(x)
x = layers.MaxPooling2D(3)(x)
x = layers.Conv2D(32, 3, activation="relu")(x)
x = layers.Conv2D(16, 3, activation="relu")(x)
encoder_output = layers.GlobalMaxPooling2D()(x)

encoder = keras.Model(encoder_input, encoder_output, name="encoder")
encoder.summary()

x = layers.Reshape((4, 4, 1))(encoder_output)
x = layers.Conv2DTranspose(16, 3, activation="relu")(x)
x = layers.Conv2DTranspose(32, 3, activation="relu")(x)
x = layers.UpSampling2D(3)(x)
x = layers.Conv2DTranspose(16, 3, activation="relu")(x)
decoder_output = layers.Conv2DTranspose(1, 3, activation="relu")(x)

autoencoder = keras.Model(encoder_input, decoder_output, name="autoencoder")
autoencoder.summary()

In [ ]:
encoder_input = keras.Input(shape=(28, 28, 1), name="original_img")
x = layers.Conv2D(16, 3, activation="relu")(encoder_input)
x = layers.Conv2D(32, 3, activation="relu")(x)
x = layers.MaxPooling2D(3)(x)
x = layers.Conv2D(32, 3, activation="relu")(x)
x = layers.Conv2D(16, 3, activation="relu")(x)
encoder_output = layers.GlobalMaxPooling2D()(x)

encoder = keras.Model(encoder_input, encoder_output, name="encoder")
encoder.summary()

decoder_input = keras.Input(shape=(16,), name="encoded_img")
x = layers.Reshape((4, 4, 1))(decoder_input)
x = layers.Conv2DTranspose(16, 3, activation="relu")(x)
x = layers.Conv2DTranspose(32, 3, activation="relu")(x)
x = layers.UpSampling2D(3)(x)
x = layers.Conv2DTranspose(16, 3, activation="relu")(x)
decoder_output = layers.Conv2DTranspose(1, 3, activation="relu")(x)

decoder = keras.Model(decoder_input, decoder_output, name="decoder")
decoder.summary()

autoencoder_input = keras.Input(shape=(28, 28, 1), name="img")
encoded_img = encoder(autoencoder_input)
decoded_img = decoder(encoded_img)
autoencoder = keras.Model(autoencoder_input, decoded_img, name="autoencoder")
autoencoder.summary()

In [ ]:
def get_model():
    inputs = keras.Input(shape=(128,))
    outputs = layers.Dense(1)(inputs)
    return keras.Model(inputs, outputs)


model1 = get_model()
model2 = get_model()
model3 = get_model()

inputs = keras.Input(shape=(128,))
y1 = model1(inputs)
y2 = model2(inputs)
y3 = model3(inputs)
outputs = layers.average([y1, y2, y3])
ensemble_model = keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
def conv(X):
    model = keras.Model(